In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/employee_activity.csv")
df.head()

,employee_id,employee_name,department,date,login_time,logout_time,ip_address,location,device_type,failed_login_attempts,files_accessed,sensitive_files_accessed,download_count,usb_usage
0,101,Arjun,IT,2026-04-01,09:10:00,17:45:00,192.168.1.10,Chennai,Laptop,1,25,2,5,0
1,102,Meena,HR,2026-04-01,08:55:00,17:30:00,192.168.1.11,Chennai,Desktop,0,15,0,2,0
2,103,Ravi,Finance,2026-04-01,10:30:00,19:00:00,192.168.1.12,Bangalore,Laptop,3,40,5,12,1
3,104,Divya,IT,2026-04-01,22:15:00,02:00:00,192.168.1.13,Chennai,Laptop,5,60,10,20,1
4,105,Karthik,Sales,2026-04-01,09:30:00,18:00:00,192.168.1.14,Mumbai,Mobile,0,10,0,1,0


In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   employee_id               10 non-null     int64
 1   employee_name             10 non-null     str  
 2   department                10 non-null     str  
 3   date                      10 non-null     str  
 4   login_time                10 non-null     str  
 5   logout_time               10 non-null     str  
 6   ip_address                10 non-null     str  
 7   location                  10 non-null     str  
 8   device_type               10 non-null     str  
 9   failed_login_attempts     10 non-null     int64
 10  files_accessed            10 non-null     int64
 11  sensitive_files_accessed  10 non-null     int64
 12  download_count            10 non-null     int64
 13  usb_usage                 10 non-null     int64
dtypes: int64(6), str(8)
memory usage: 1.2 KB


,employee_id,failed_login_attempts,files_accessed,sensitive_files_accessed,download_count,usb_usage
count,10.00000,10.00000,10.000000,10.000000,10.000000,10.000000
mean,105.50000,1.80000,30.200000,3.400000,8.000000,0.300000
std,3.02765,2.20101,23.049705,5.168279,9.797959,0.483046
min,101.00000,0.00000,10.000000,0.000000,1.000000,0.000000
25%,103.25000,0.00000,15.750000,0.000000,2.000000,0.000000
50%,105.50000,1.00000,21.000000,1.000000,3.500000,0.000000
75%,107.75000,2.75000,36.250000,4.250000,10.250000,0.750000
max,110.00000,6.00000,80.000000,15.000000,30.000000,1.000000


In [4]:
df.isnull().sum()

employee_id                 0
employee_name               0
department                  0
date                        0
login_time                  0
logout_time                 0
ip_address                  0
location                    0
device_type                 0
failed_login_attempts       0
files_accessed              0
sensitive_files_accessed    0
download_count              0
usb_usage                   0
dtype: int64

In [5]:
df['date'] = pd.to_datetime(df['date'])
df['login_time'] = pd.to_datetime(df['login_time'], format='%H:%M:%S').dt.time
df['logout_time'] = pd.to_datetime(df['logout_time'], format='%H:%M:%S').dt.time

In [6]:
df['login_datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['login_time'].astype(str))
df['logout_datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['logout_time'].astype(str))

# Handle overnight login (logout next day)
df.loc[df['logout_datetime'] < df['login_datetime'], 'logout_datetime'] += pd.Timedelta(days=1)

df['session_duration_hours'] = (df['logout_datetime'] - df['login_datetime']).dt.total_seconds() / 3600

In [7]:
df['after_hours'] = df['login_datetime'].dt.hour.apply(lambda x: 1 if x < 8 or x > 20 else 0)

In [8]:
df = df.drop_duplicates()

In [9]:
df.head()

,employee_id,employee_name,department,date,login_time,logout_time,ip_address,location,device_type,failed_login_attempts,files_accessed,sensitive_files_accessed,download_count,usb_usage,login_datetime,logout_datetime,session_duration_hours,after_hours
0,101,Arjun,IT,2026-04-01,09:10:00,17:45:00,192.168.1.10,Chennai,Laptop,1,25,2,5,0,2026-04-01 09:10:00,2026-04-01 17:45:00,8.583333,0
1,102,Meena,HR,2026-04-01,08:55:00,17:30:00,192.168.1.11,Chennai,Desktop,0,15,0,2,0,2026-04-01 08:55:00,2026-04-01 17:30:00,8.583333,0
2,103,Ravi,Finance,2026-04-01,10:30:00,19:00:00,192.168.1.12,Bangalore,Laptop,3,40,5,12,1,2026-04-01 10:30:00,2026-04-01 19:00:00,8.500000,0
3,104,Divya,IT,2026-04-01,22:15:00,02:00:00,192.168.1.13,Chennai,Laptop,5,60,10,20,1,2026-04-01 22:15:00,2026-04-02 02:00:00,3.750000,1
4,105,Karthik,Sales,2026-04-01,09:30:00,18:00:00,192.168.1.14,Mumbai,Mobile,0,10,0,1,0,2026-04-01 09:30:00,2026-04-01 18:00:00,8.500000,0


In [10]:
df.to_csv("../data/processed/cleaned_employee_activity.csv", index=False)